# 02 · Pretrain

Trains one model of the ladder on the tokenized corpus and keeps its checkpoints in a **private** Hugging Face model repo.

| `RUN` | Model | Tokens | Steps (262,144 tokens each) | A100 time (measured tok/s) |
|---|---|---:|---:|---:|
| `smoke` | s-50m | 39M | 150 | ~10 min incl. compile |
| `s-50m` | s-50m | 1.06B (20 / param) | 4,044 | ~2.6 h |
| `m-100m` | m-100m | 1.92B (20 / param) | 7,325 | ~6.6 h |

**Runtime:** *Select Kernel → Colab → New Colab Server → **A100*** (L4 works too, ~4× slower).

**Resumable:** a checkpoint is pushed every 25 minutes and at the end. If the session dies, reconnect and run all cells again with the same `RUN`: training continues from the latest checkpoint with the same data order. Up to 25 minutes of work can be lost.

**Order:** run `smoke` first. If it finishes cleanly, change `RUN` and run the cells again.

## 1. GPU

In [ ]:
import subprocess

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"],
                     capture_output=True, text=True).stdout)

## 2. Get the code

In [ ]:
import os
import sys
from pathlib import Path

REPO_URL = "https://github.com/Tbt-code/GLM-Ladder-from-Scratch.git"
BRANCH = "main"
REPO_DIR = Path("/content/glm-ladder-from-scratch")

if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--quiet", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--quiet", "--hard", f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "--quiet", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-r", str(REPO_DIR / "requirements.txt")], check=True)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print("commit:", commit)

## 3. Run all tests

The resume test in `tests/test_pretrain.py` is the one that matters most here.

In [ ]:
result = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], capture_output=True, text=True)
print(result.stdout[-2500:], result.stderr[-1000:])
assert result.returncode == 0, "tests failed - fix before training"

## 4. Log in to Hugging Face

The token is typed into a hidden prompt and lives only in this session's memory. It must be able to read `glm-ladder-data` and write to `glm-ladder-runs`. Never paste it into a cell.

In [ ]:
from getpass import getpass

from huggingface_hub import HfApi

DATA_REPO = "Tbt-AI/glm-ladder-data"
RUNS_REPO = "Tbt-AI/glm-ladder-runs"

token = getpass("Hugging Face write token: ")
api = HfApi(token=token)
print("logged in as:", api.whoami()["name"])
for repo_id, repo_type in ((DATA_REPO, "dataset"), (RUNS_REPO, "model")):
    assert api.repo_info(repo_id, repo_type=repo_type).private, f"{repo_id} must be private"
    print("private:", repo_id)

## 5. Download and verify the corpus

About 4.4 GB. Every shard's SHA-256 is checked against its manifest, and every manifest's tokenizer hash against the tokenizer receipt in git. A rerun in the same session skips the download.

In [ ]:
import json
import time

from huggingface_hub import snapshot_download

from glm_ladder.checkpoint import sha256_file

DATA_DIR = Path("/content/data")
started = time.perf_counter()
snapshot_download(repo_id=DATA_REPO, repo_type="dataset", local_dir=DATA_DIR, token=token,
                  allow_patterns=["*/manifest.json", "*/*.bin"])
print(f"downloaded in {time.perf_counter() - started:.0f} s")

tokenizer_receipt = json.loads((REPO_DIR / "artifacts" / "receipts" / "tokenizer-16k.json").read_text())
corpus_receipt = json.loads((REPO_DIR / "artifacts" / "receipts" / "corpus-manifest.json").read_text())
for source in corpus_receipt["sources"]:
    manifest = json.loads((DATA_DIR / source / "manifest.json").read_text())
    assert manifest["tokenizer_sha256"] == tokenizer_receipt["sha256"], f"{source}: tokenizer mismatch"
    for shard in manifest["shards"]:
        path = DATA_DIR / source / shard["name"]
        assert path.stat().st_size == 2 * shard["tokens"], f"{path}: wrong size"
        assert sha256_file(path) == shard["sha256"], f"{path}: SHA-256 mismatch"
    print(f"{source:8s} {len(manifest['shards']):2d} shards ok")
print(f"verified in {time.perf_counter() - started:.0f} s")

## 6. Data sources and mix

The 60 / 30 / 10 mix was defined in bytes. Turkish and code need more tokens per byte than English, so the token weights differ slightly.

In [ ]:
from glm_ladder.data_sources import MIX
from glm_ladder.loader import PackedSource, token_mix

SEQ_LEN = 1024


def shard_paths(source: str, prefix: str) -> list[str]:
    return sorted(str(path) for path in (DATA_DIR / source).glob(f"{prefix}-*.bin"))


train_sources = {source: PackedSource(shard_paths(source, "train"), SEQ_LEN) for source in MIX}
val_sources = {source: PackedSource(shard_paths(source, "val"), SEQ_LEN) for source in MIX}
weights = token_mix(MIX, tokenizer_receipt["bytes_per_token"])
for source in MIX:
    print(f"{source:8s} weight {weights[source]:.3f}  train windows {train_sources[source].num_windows:,}  "
          f"val windows {val_sources[source].num_windows:,}")

## 7. Choose the run

Change only `RUN`. A run name that already has checkpoints in `glm-ladder-runs` **resumes**; to start that experiment again from zero, bump the suffix (`-002`).

In [ ]:
import torch

from glm_ladder import get_preset
from glm_ladder.pretrain import PretrainConfig

RUN = "s-50m"            # "smoke" -> "s-50m" -> "m-100m"

GPU = torch.cuda.get_device_name()
MICRO_BATCH = 32 if "A100" in GPU else 16            # 32 x 1024 fits in 40 GB for both models
common = dict(seq_len=SEQ_LEN, micro_batch=MICRO_BATCH, compile=True)
RUNS = {
    "smoke": ("s-50m", PretrainConfig(run_name="smoke-s50m-001", total_tokens=150 * 262_144, warmup_steps=20,
                                      eval_every=50, checkpoint_every_steps=75, **common)),
    "s-50m": ("s-50m", PretrainConfig(run_name="s50m-001", total_tokens=1_060_000_000, **common)),
    "m-100m": ("m-100m", PretrainConfig(run_name="m100m-001", total_tokens=1_920_000_000, **common)),
}
preset, cfg = RUNS[RUN]
model_config = get_preset(preset)
measured = {"s-50m": 166_500, "m-100m": 117_000}[preset]   # A100 tok/s; s-50m measured, m-100m estimated (x1.42)
print(f"{cfg.run_name}: {preset}, {cfg.total_steps:,} steps x {cfg.tokens_per_step:,} tokens, "
      f"accumulation {cfg.batch_rows // cfg.micro_batch}, on {GPU}")
print(f"estimate on A100: {cfg.total_steps * cfg.tokens_per_step / measured / 3600:.1f} h")

## 8. Train

Expect the first step to take a minute or two (`torch.compile`), and the loss to start near ln(16384) ≈ 9.70.

**Optional resume drill (smoke run only):** stop this cell after the checkpoint at step 75 is pushed, then run it again. It should print `resumed ... step 75` and continue.

In [ ]:
from glm_ladder.pretrain import run_pretraining
from glm_ladder.runs import RunStore

WORKDIR = Path("/content/runs")
store = RunStore(api, RUNS_REPO, cfg.run_name)
metadata = {"commit": commit, "preset": preset, "gpu": GPU, "data_repo": DATA_REPO,
            "tokenizer_sha256": tokenizer_receipt["sha256"], "weights": weights}
result = run_pretraining(cfg, model_config, train_sources=train_sources, val_sources=val_sources,
                         weights=weights, store=store, workdir=WORKDIR, device=torch.device("cuda"),
                         metadata=metadata)
print("finished:", result["finished"], "at step", result["step"])

## 9. Curves

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from glm_ladder.runs import MetricsLog

rows = MetricsLog(WORKDIR / cfg.run_name / "metrics.jsonl").rows()
train = [row for row in rows if "loss" in row]
evals = [row for row in rows if "eval" in row]
tokens = np.array([row["tokens"] for row in train]) / 1e6
loss = np.array([row["loss"] for row in train])
smooth = np.convolve(loss, np.ones(20) / 20, mode="valid") if len(loss) >= 20 else loss

fig, (left, right) = plt.subplots(1, 2, figsize=(13, 4))
left.plot(tokens, loss, alpha=0.25, label="train")
left.plot(tokens[len(tokens) - len(smooth):], smooth, label="train (20-step mean)")
left.plot([row["step"] * cfg.tokens_per_step / 1e6 for row in evals], [row["eval_mixed"] for row in evals],
          "o-", label="val (mixed)")
left.set(xlabel="tokens (M)", ylabel="cross-entropy", title=cfg.run_name)
left.legend()
for source in MIX:
    right.plot([row["step"] * cfg.tokens_per_step / 1e6 for row in evals], [row["eval"][source] for row in evals],
               "o-", label=source)
right.set(xlabel="tokens (M)", ylabel="val cross-entropy", title="validation by source")
right.legend()
plt.tight_layout()
plt.show()

## 10. Receipt

Copy the printed JSON into `artifacts/receipts/pretrain-<run_name>.json` in VS Code and commit it. Weights stay in the Hugging Face repo.

In [ ]:
from dataclasses import asdict

final = evals[-1] if evals else {}
seconds = sum(row["seconds"] for row in train)
receipt = {
    "schema_version": "1.0", "kind": "pretraining_run", "run_name": cfg.run_name,
    "commit": commit, "preset": preset, "gpu": GPU,
    "parameters": result["model"].parameter_counts(),
    "pretrain_config": asdict(cfg),
    "steps": result["step"], "tokens_seen": result["tokens_seen"], "finished": result["finished"],
    "train_hours": round(seconds / 3600, 2),
    "mean_tokens_per_second": round(result["tokens_seen"] / seconds) if seconds else None,
    "final_eval": final.get("eval"), "final_eval_mixed": final.get("eval_mixed"),
    "checkpoints": f"{RUNS_REPO}/runs/{cfg.run_name} @ {store.latest()}",
}
print(json.dumps(receipt, indent=2))